
# 15 — Prompt Engineering for AI Agents with Human Oversight

**Learning reference:**
[BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Topics adapted from the course:**
- AI agent tool calling
- LangGraph `ToolNode`
- Stateful workflows
- Human-in-the-loop approval
- `interrupt()` and `Command(resume=...)`
- Checkpointing and execution history
- Controlled execution of model-proposed actions

**LangGraph documentation:**
https://docs.langchain.com/oss/python/langgraph/overview

## Goal

Build a small AI-assisted customer-support workflow.

Gemini receives a customer request and proposes a
policy-lookup tool call using a carefully designed prompt.

LangGraph pauses before the tool executes.

A human can then approve or reject the proposal.

We will test two scenarios:

1. An approved proposal that executes the tool.
2. A rejected proposal that never executes the tool.

The example uses fictional local policies and
does not modify any external system.

The purpose is to demonstrate how prompt engineering,
tool calling and human oversight work together.



## 1. From LLM Workflows to AI Agents

In Notebook 13, Gemini generated grounded answers.

In Notebook 14, Gemini selected a category that
determined which graph branch executed.

In this notebook, Gemini proposes a **tool call**.

A tool call is a structured request to execute
a named function with specific arguments.

For example:

Tool: `lookup_policy`

Arguments:
`{"category": "damage"}`

A tool call proposal is not the same as executing
the tool.

The application decides whether and when the
requested function should run.

We will place a human approval step between
the model proposal and tool execution.

This is especially important for agents with
tools that can change files, send messages or
perform transactions.

Our example uses a read-only policy lookup
so the experiment remains safe and reproducible.


In [17]:

%pip install -q "langgraph>=1,<2" "langchain-core>=1,<2" "langchain-openai>=1,<2"


In [18]:

from typing import Annotated, Literal, TypedDict
from operator import add

from langchain_core.messages import (
    AIMessage,
    ToolMessage
)
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import ToolNode
from langgraph.types import interrupt, Command


POLICIES = {
    "returns": (
        "Unused products may be returned within "
        "30 days with a receipt."
    ),
    "delivery": (
        "Standard delivery usually takes "
        "3 to 5 business days. "
        "Contact support about delayed parcels."
    ),
    "damage": (
        "Customers who receive damaged products "
        "should contact support to discuss "
        "replacement options."
    ),
    "general": (
        "Contact customer support for assistance "
        "with other questions."
    )
}


class AgentState(TypedDict, total=False):
    request: str
    messages: Annotated[list, add_messages]
    approved: bool
    outcome: str
    log: Annotated[list[str], add]


@tool
def lookup_policy(
    category: Literal[
        "returns",
        "delivery",
        "damage",
        "general"
    ]
) -> str:
    """Retrieve the fictional support policy for a category.

    Use returns for returns or refunds.
    Use delivery for shipping and missing parcels.
    Use damage for broken or damaged products.
    Use general for other support requests.
    """
    return POLICIES[category]


print("Tool defined:", lookup_policy.name)
print("Allowed categories:", list(POLICIES))
print("Model requests: 0")


Tool defined: lookup_policy
Allowed categories: ['returns', 'delivery', 'damage', 'general']
Model requests: 0



## 2. Prompt Engineering for Tool Selection

The model must decide whether a customer request
concerns returns, delivery, damage or another issue.

Instead of asking Gemini to return a category label,
we provide an actual tool definition.

The model must propose one call to `lookup_policy`
with the appropriate category argument.

Our instructions specify:

- Which tool to use.
- What the tool is for.
- How to select the category.
- That only one tool call should be proposed.
- That the model must not pretend the tool has run.
- That the model must not invent policy information.

The tool schema further constrains the argument
to four allowed categories.

The application will validate the proposal before
passing it to the human-review step.

The model can propose an action, but cannot bypass
the approval process defined by LangGraph.


In [19]:

from getpass import getpass

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


MODEL = "gemini-3.5-flash-lite"

gemini_key = getpass("Enter your Gemini API key: ")

gemini = ChatOpenAI(
    model=MODEL,
    api_key=gemini_key,
    base_url=(
        "https://generativelanguage.googleapis.com/"
        "v1beta/openai/"
    ),
    temperature=0,
    max_retries=0,
    timeout=120,
    use_responses_api=False
)


agent_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a customer-support tool-selection assistant.

Your task is to choose the policy lookup needed
for the customer's request.

You have one tool: lookup_policy.

Use it exactly once with one category:

- returns: returns and refund requests
- delivery: shipping, delayed or missing parcels
- damage: damaged or broken products
- general: other customer-support questions

Rules:
1. Propose exactly one lookup_policy tool call.
2. Select the category describing the main issue.
3. Do not write a final customer-support answer.
4. Do not invent policy information.
5. Do not claim that the tool has already executed.
6. The application handles human approval.

A tool proposal must be made before the workflow
can proceed.
"""
    ),
    (
        "human",
        "Customer request:\n{request}"
    )
])


tool_enabled_model = gemini.bind_tools(
    [lookup_policy]
)

proposal_chain = agent_prompt | tool_enabled_model

print("Gemini tool-calling chain configured.")
print("Model requests: 0")


Enter your Gemini API key: ··········
Gemini tool-calling chain configured.
Model requests: 0



## 3. Build the Human-Approval Workflow

Our LangGraph agent contains four steps.

**Propose**

Gemini produces a structured tool-call request.

**Review**

LangGraph uses `interrupt()` to pause execution.
A human must explicitly approve or reject the call.

**Execute**

If approved, `ToolNode` executes the proposed
`lookup_policy` function.

**Finalize**

The graph records whether the action was executed
and what result was returned.

Rejection bypasses the tool entirely.

We use `InMemorySaver` to preserve workflow state
between the initial execution and resumption.

Each test case will use a different `thread_id`
so its history is kept separate.

We will also inspect checkpoint history after
both workflows finish.


In [20]:

def propose_tool_call(state: AgentState):
    response = proposal_chain.invoke({
        "request": state["request"]
    })

    if response.invalid_tool_calls:
        raise ValueError(
            "Gemini returned malformed tool-call data."
        )

    if len(response.tool_calls) != 1:
        raise ValueError(
            "Expected exactly one Gemini tool call, "
            f"received {len(response.tool_calls)}."
        )

    call = response.tool_calls[0]

    if call["name"] != "lookup_policy":
        raise ValueError(
            f"Unexpected tool: {call['name']}"
        )

    category = call["args"].get("category")

    if category not in POLICIES:
        raise ValueError(
            f"Invalid tool category: {category!r}"
        )

    return {
        "messages": [response],
        "log": [
            f"Gemini proposed lookup_policy({category})"
        ]
    }


def human_review(state: AgentState):
    proposed_call = state["messages"][-1].tool_calls[0]

    review_request = {
        "message": "Human approval required",
        "tool": proposed_call["name"],
        "arguments": proposed_call["args"],
        "allowed_decisions": ["approve", "reject"]
    }

    decision = interrupt(review_request)

    if decision not in ("approve", "reject"):
        raise ValueError(
            "Human decision must be approve or reject."
        )

    approved = decision == "approve"

    return {
        "approved": approved,
        "log": [f"Human decision: {decision}"]
    }


def route_after_review(state: AgentState):
    if state["approved"]:
        return "execute"

    return "reject"


def finalize_approved(state: AgentState):
    tool_messages = [
        message
        for message in state["messages"]
        if isinstance(message, ToolMessage)
    ]

    if not tool_messages:
        raise ValueError(
            "Approved workflow has no tool result."
        )

    result = tool_messages[-1]

    if result.status == "error":
        raise RuntimeError(
            f"Tool execution failed: {result.content}"
        )

    return {
        "outcome": str(result.content),
        "log": ["Approved tool call executed"]
    }


def finalize_rejected(state: AgentState):
    return {
        "outcome": (
            "Rejected by human reviewer. "
            "No policy lookup was executed."
        ),
        "log": ["Tool execution skipped"]
    }


print("Agent nodes defined.")
print("Model requests: 0")


Agent nodes defined.
Model requests: 0


In [21]:

builder = StateGraph(AgentState)

builder.add_node("propose", propose_tool_call)
builder.add_node("review", human_review)

builder.add_node(
    "execute",
    ToolNode([lookup_policy])
)

builder.add_node(
    "approved_result",
    finalize_approved
)

builder.add_node(
    "rejected_result",
    finalize_rejected
)

builder.add_edge(START, "propose")
builder.add_edge("propose", "review")

builder.add_conditional_edges(
    "review",
    route_after_review,
    {
        "execute": "execute",
        "reject": "rejected_result"
    }
)

builder.add_edge(
    "execute",
    "approved_result"
)

builder.add_edge("approved_result", END)
builder.add_edge("rejected_result", END)

checkpointer = InMemorySaver()

agent_graph = builder.compile(
    checkpointer=checkpointer
)

print("Human-controlled AI agent compiled.")
print("No Gemini requests made yet.")


Human-controlled AI agent compiled.
No Gemini requests made yet.



## 4. Case A — Approve Gemini's Tool Proposal

Customer request:

"My new headphones arrived broken."

Expected proposed category: `damage`

We will run the graph until the human-review
interrupt.

At that point:

- Gemini has proposed a tool call.
- The graph has saved its execution state.
- The lookup tool has not executed.

We will inspect the proposal and then explicitly
approve it.

After approval, LangGraph will execute the
tool through `ToolNode`.

This example makes one Gemini model request.
Resuming after the interrupt does not need
another model request.


In [22]:

case_a_config = {
    "configurable": {
        "thread_id": "agent-approval-case"
    }
}

case_a_request = (
    "My new headphones arrived broken."
)

if "case_a_paused" not in globals():
    case_a_paused = agent_graph.invoke(
        {
            "request": case_a_request,
            "messages": [],
            "log": []
        },
        config=case_a_config
    )

    print("Gemini proposal generated.")
else:
    print("Using existing saved proposal.")


assert "__interrupt__" in case_a_paused

review = case_a_paused["__interrupt__"][0].value

print("\nCustomer request:", case_a_request)
print("Proposed tool:", review["tool"])
print("Proposed arguments:", review["arguments"])
print("Approval required:", True)

snapshot = agent_graph.get_state(case_a_config)

assert not any(
    isinstance(message, ToolMessage)
    for message in snapshot.values["messages"]
)

print("\nTool executed before approval: NO")


Gemini proposal generated.

Customer request: My new headphones arrived broken.
Proposed tool: lookup_policy
Proposed arguments: {'category': 'damage'}
Approval required: True

Tool executed before approval: NO


In [23]:

if "case_a_result" not in globals():
    case_a_result = agent_graph.invoke(
        Command(resume="approve"),
        config=case_a_config
    )

    print("Human approval applied.")
else:
    print("Using saved approved result.")


print("\nApproved:", case_a_result["approved"])
print("Outcome:", case_a_result["outcome"])

print("\nExecution log:")
for entry in case_a_result["log"]:
    print("-", entry)

tool_results = [
    message
    for message in case_a_result["messages"]
    if isinstance(message, ToolMessage)
]

assert case_a_result["approved"] is True
assert len(tool_results) == 1

print("\nToolNode execution: VERIFIED")
print("Additional Gemini requests: 0")


Human approval applied.

Approved: True
Outcome: Customers who receive damaged products should contact support to discuss replacement options.

Execution log:
- Gemini proposed lookup_policy(damage)
- Human decision: approve
- Approved tool call executed

ToolNode execution: VERIFIED
Additional Gemini requests: 0



## 5. Case B — Reject Gemini's Tool Proposal

Customer request:

"I want to return an unused jacket."

Expected proposed category: `returns`

This time, the human reviewer will reject the
proposed tool call.

The graph should:

1. Obtain a genuine proposal from Gemini.
2. Pause before executing the lookup.
3. Record the rejection.
4. Skip the ToolNode entirely.
5. Finish without a tool result.

This demonstrates that an AI-generated tool call
does not automatically authorize execution.

The rejection applies even if Gemini's proposed
category is correct.


In [24]:

case_b_config = {
    "configurable": {
        "thread_id": "agent-rejection-case"
    }
}

case_b_request = (
    "I want to return an unused jacket."
)

if "case_b_paused" not in globals():
    case_b_paused = agent_graph.invoke(
        {
            "request": case_b_request,
            "messages": [],
            "log": []
        },
        config=case_b_config
    )

    print("Gemini proposal generated.")
else:
    print("Using existing saved proposal.")


assert "__interrupt__" in case_b_paused

review_b = case_b_paused["__interrupt__"][0].value

print("\nCustomer request:", case_b_request)
print("Proposed tool:", review_b["tool"])
print("Proposed arguments:", review_b["arguments"])

snapshot_b = agent_graph.get_state(case_b_config)

assert not any(
    isinstance(message, ToolMessage)
    for message in snapshot_b.values["messages"]
)

print("\nTool executed before approval: NO")


Gemini proposal generated.

Customer request: I want to return an unused jacket.
Proposed tool: lookup_policy
Proposed arguments: {'category': 'returns'}

Tool executed before approval: NO


In [25]:

if "case_b_result" not in globals():
    case_b_result = agent_graph.invoke(
        Command(resume="reject"),
        config=case_b_config
    )

    print("Human rejection applied.")
else:
    print("Using saved rejected result.")


print("\nApproved:", case_b_result["approved"])
print("Outcome:", case_b_result["outcome"])

print("\nExecution log:")
for entry in case_b_result["log"]:
    print("-", entry)

rejected_tool_results = [
    message
    for message in case_b_result["messages"]
    if isinstance(message, ToolMessage)
]

assert case_b_result["approved"] is False
assert len(rejected_tool_results) == 0

print("\nTool execution correctly skipped.")
print("Additional Gemini requests: 0")


Human rejection applied.

Approved: False
Outcome: Rejected by human reviewer. No policy lookup was executed.

Execution log:
- Gemini proposed lookup_policy(returns)
- Human decision: reject
- Tool execution skipped

Tool execution correctly skipped.
Additional Gemini requests: 0



## 6. Evaluate AI Proposals and Checkpoint History

We will now verify both executions without
making further model requests.

The evaluation checks:

- Whether Gemini proposed the expected category.
- Whether both cases paused for human approval.
- Whether the approved case executed the tool.
- Whether the rejected case skipped execution.
- Whether LangGraph retained separate execution
  histories for the two threads.

Checkpointing is important because human approval
may happen after the model has already produced
its tool call.

The checkpointer preserves workflow state so
execution can resume without asking Gemini to
generate the proposal again.

We use in-memory checkpointing here. Persistent
storage, such as SQLite, is possible but is not
needed for this small experiment.


In [26]:

case_a_category = (
    case_a_paused["__interrupt__"][0]
    .value["arguments"]["category"]
)

case_b_category = (
    case_b_paused["__interrupt__"][0]
    .value["arguments"]["category"]
)

category_checks = {
    "Approved case classification": (
        case_a_category == "damage"
    ),
    "Rejected case classification": (
        case_b_category == "returns"
    )
}

approved_executed = (
    case_a_result["approved"] is True
    and len(tool_results) == 1
    and case_a_result["outcome"]
        == POLICIES[case_a_category]
)

rejected_skipped = (
    case_b_result["approved"] is False
    and len(rejected_tool_results) == 0
)

history_a = list(
    agent_graph.get_state_history(case_a_config)
)

history_b = list(
    agent_graph.get_state_history(case_b_config)
)

checkpoint_checks = {
    "Approval history saved": len(history_a) > 1,
    "Rejection history saved": len(history_b) > 1
}

all_checks = {
    **category_checks,
    "Approved tool executed": approved_executed,
    "Rejected tool skipped": rejected_skipped,
    **checkpoint_checks
}

for name, passed in all_checks.items():
    print(
        f"{name}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

passed_total = sum(all_checks.values())

print(
    f"\nEvaluation: "
    f"{passed_total}/{len(all_checks)} checks passed"
)

print("\nApproval checkpoints:", len(history_a))
print("Rejection checkpoints:", len(history_b))

assert approved_executed
assert rejected_skipped
assert all(checkpoint_checks.values())

if all(category_checks.values()):
    print("Both model proposals matched expected categories.")
else:
    print(
        "At least one model classification differed "
        "from the expected category."
    )

print("\nAdditional Gemini requests: 0")


Approved case classification: PASS
Rejected case classification: PASS
Approved tool executed: PASS
Rejected tool skipped: PASS
Approval history saved: PASS
Rejection history saved: PASS

Evaluation: 6/6 checks passed

Approval checkpoints: 6
Rejection checkpoints: 5
Both model proposals matched expected categories.

Additional Gemini requests: 0



## Findings and Limitations

### What This Notebook Demonstrates

- Designing prompts for tool-using language models.
- Connecting Gemini to a real LangChain tool.
- Using model-generated tool calls instead of
  manually simulated tool-call messages.
- Inspecting and validating the proposed tool.
- Building an AI agent with LangGraph.
- Pausing execution with `interrupt()`.
- Resuming execution using `Command(resume=...)`.
- Executing an approved tool through `ToolNode`.
- Preventing rejected tool calls from executing.
- Preserving state using `InMemorySaver`.
- Inspecting checkpoints for separate execution threads.
- Testing model decisions and workflow controls.

### Prompt Engineering Techniques

**Tool-use instructions**

The system prompt tells Gemini when and how to
propose the available policy-lookup tool.

**Constrained arguments**

The tool schema defines the allowed policy
categories.

**Explicit behavioral constraints**

The prompt prevents the model from presenting a
tool proposal as an already completed action.

**Validation**

The application rejects missing, malformed or
unrecognized tool proposals.

**Human oversight**

Model-generated tool calls require explicit
approval before execution.

### Relationship to Earlier Notebooks

**Notebook 13 — LangChain and LCEL**

Gemini generates answers grounded in retrieved
policy information.

**Notebook 14 — AI-Driven LangGraph**

Gemini classifies requests to determine which
workflow branch executes.

**Notebook 15 — Human-Controlled AI Agents**

Gemini proposes a tool call, but a human controls
whether the tool executes.

Together, these examples progress from LLM
generation to model-driven workflow decisions
and controlled agent actions.

### Observed Results

Two Gemini-generated tool proposals were tested
using the LangGraph human-approval workflow.

**Case A — Approved**

- Request: "My new headphones arrived broken."
- Expected category: `damage`.
- Gemini proposed: `lookup_policy(category="damage")`.
- Execution paused before the tool call.
- Human decision: `approve`.
- ToolNode executed the policy lookup.
- Six checkpoints were recorded.

**Case B — Rejected**

- Request: "I want to return an unused jacket."
- Expected category: `returns`.
- Gemini proposed: `lookup_policy(category="returns")`.
- Execution paused before the tool call.
- Human decision: `reject`.
- No ToolNode execution occurred.
- Five checkpoints were recorded.

**Evaluation summary**

- Model classifications: 2/2 matched expected categories.
- Approved tool execution: verified.
- Rejected tool execution: successfully prevented.
- Separate execution histories: verified.
- Overall evaluation: 6/6 checks passed.

The experiment demonstrates controlled execution
of real model-proposed tool calls on two examples.

It does not establish reliability across other
requests, tools or approval decisions.

### Limitations

- The example uses only two fictional customer requests.
- The policy-lookup tool is read-only.
- The human decision is supplied manually in code;
  there is no interactive approval interface.
- The prompt requests one tool call, but output is
  validated because model compliance is not guaranteed.
- The demonstration covers approval and rejection,
  not editing proposed tool arguments.
- It does not implement multi-step agent planning.
- Checkpoints are stored only in memory.
- The workflow does not perform real transactions.
- Model classification accuracy cannot be generalized
  from two test examples.

### Conclusion

This notebook demonstrates how prompt engineering
and human-controlled AI agents can work together.

The prompt defines the model's tool-selection task.

Gemini proposes a structured tool call.

LangGraph pauses execution and preserves state.

A human decision determines whether ToolNode
executes the requested function.

The main principle is that a language model may
propose an action without having authority to
execute it.
